# Self-Consistency: Sample-and-Vote Reasoning

Welcome to this notebook on **Self-Consistency**, one of the simplest yet most effective reliability upgrades you can bolt onto a plain Chain-of-Thought (CoT) prompt. This pattern is inspired by the `21_self_consistency` concept in FareedKhan-dev's [`all-agentic-architectures`](https://github.com/FareedKhan-dev/all-agentic-architectures) collection.

A single call to an LLM with Chain-of-Thought prompting is a **single sample from a distribution over possible reasoning paths**. At temperature 0 that distribution collapses to one path, which may or may not be correct. At a non-zero temperature, re-asking the same question produces *different* reasoning paths — some correct, some flawed in different ways.

Self-Consistency exploits this: instead of trusting one sampled path, it draws **N independent samples**, lets each one reason through the problem on its own (no path sees any other path), extracts each sample's final answer, and then takes a **majority vote** across the N final answers. Because independent errors rarely agree with each other while the correct reasoning path tends to be reproducible, the vote washes out noise and converges on the right answer far more reliably than any single sample.

To make the payoff concrete, we'll run this on a math word problem with a **single verifiable numeric answer**, so the majority vote is easy to check by hand.

| Property | Value |
|---|---|
| Origin | Wang et al. (Google), *Self-Consistency Improves Chain of Thought Reasoning in Language Models* (2022). [arXiv:2203.11171](https://arxiv.org/abs/2203.11171) |


### Definition
**Self-Consistency** is a decoding-time reasoning strategy where the same prompt (typically a Chain-of-Thought prompt) is sampled **N times independently at a non-zero temperature**, each sample producing its own full reasoning trace and final answer. The final answers are then aggregated by **majority vote** (or another consolidation rule), and the most common answer is returned as the consolidated result.

### High-level Workflow

1. **Fix the question, vary the sampling.** The exact same CoT prompt is sent to the LLM `N` times (e.g. `N=5`), with `temperature > 0` so each call can explore a different reasoning path.
2. **Independent reasoning.** Each of the `N` calls reasons through the problem from scratch. Crucially, no sample sees any other sample's reasoning — there is no cross-talk, unlike a debate or reflection loop.
3. **Extract the final answer.** From each sample's free-form reasoning, pull out just the final answer (e.g. via an instructed format like `Final Answer: <value>`, or a structured-output schema).
4. **Vote.** Tally how often each distinct final answer occurs across the `N` samples.
5. **Consolidate.** Return the answer with the most votes (plurality/majority). Ties can be broken arbitrarily, by re-sampling, or by falling back to the highest-temperature-0 answer.

### When to Use / Applications
- **Problems with a single, checkable final answer:** math word problems, arithmetic, unit conversions, multiple-choice questions, short-answer factual questions.
- **Cheap way to boost accuracy without training:** no fine-tuning, no verifier model, no tool use — just repeated sampling plus a vote.
- **Reducing variance from a single unlucky sample:** guards against the case where one CoT trace goes down a locally plausible but wrong reasoning branch.
- **A building block inside larger systems:** e.g. as the "propose" step feeding into Tree-of-Thoughts, LATS, or an Ensemble aggregator (see Layer B4 below).

### Strengths
- Extremely simple to implement — no new architecture, just a loop and a `Counter`.
- No extra model or judge is required; the "verifier" is just agreement among peers.
- Reliably improves accuracy on reasoning tasks compared to single-sample CoT, especially when the failure modes of individual samples are uncorrelated.
- Fully parallelizable — the `N` samples are independent and can be fired off concurrently.

### Weaknesses
- **N times the cost and latency** of a single call — there is no free lunch.
- Only works when a final answer can be **cleanly extracted and compared** (numeric or short-string answers vote well; long free-form essays do not).
- If the model has a **systematic bias** (the same mistake in most samples), majority voting will confidently converge on the *wrong* answer — self-consistency corrects for *random* errors, not *systematic* ones.
- Does not explore alternative strategies the way Tree-of-Thoughts does; each sample follows its own path but there's no branching/backtracking or evaluation of intermediate steps.

## Phase 0: Foundation & Setup

This repo routes all LLM calls through the shared `helpers` factory rather than instantiating provider clients directly. `get_llm()` is platform-aware (Groq on Windows, Databricks on macOS by default) and exposes `temperature` as a plain keyword argument — exactly what we need to get non-zero-temperature, independent samples for Self-Consistency.

In [ ]:
# ============ IMPORTS & ENVIRONMENT ============
import re
from collections import Counter

from helpers import get_llm

# `get_llm` signature (see helpers/utils.py):
#   get_llm(*, provider=None, model=None, temperature=0, verbose=True)
# temperature is a supported keyword — we use it to get diverse, independent
# reasoning paths across the N samples of the same question.


### What We Are Going to Do

1. Pose a single math word problem that has one correct numeric answer.
2. Write one Chain-of-Thought prompt that asks the model to reason step by step and end with a line `Final Answer: <number>`.
3. Run a **single-sample baseline** at a non-zero temperature — this is what a naive "ask once" CoT agent would return, and it may or may not be correct.
4. Run the **same prompt N=5 times independently** at the same non-zero temperature to build our Self-Consistency ensemble.
5. Extract each sample's `Final Answer`, tally the votes with `collections.Counter`, and report the majority-vote answer.
6. Compare the majority-vote answer against the ground truth and against the single-sample baseline to make the value of sampling-and-voting visible.

In [ ]:
# ============ PROBLEM DEFINITION ============
# A math word problem with a single, easily verifiable numeric answer.
# It's deliberately "trap-y": the afternoon sales are defined relative to the
# morning sales, and the trays are added mid-story, so a rushed reasoning path
# can misplace a term or apply an operation in the wrong order.

QUESTION = (
    "A bakery starts the day with 23 cupcakes on the shelf. Partway through the "
    "morning they bake 4 more trays, each holding 6 cupcakes. They sell 15 "
    "cupcakes in the morning, and in the afternoon they sell twice as many "
    "cupcakes as they sold in the morning. How many cupcakes are left at the "
    "end of the day?"
)

# Ground truth, worked out independently, used only to grade our results below:
#   starting: 23
#   + baked:   4 trays * 6 = 24            -> 23 + 24 = 47
#   - morning sales: 15                     -> 47 - 15 = 32
#   - afternoon sales: 2 * 15 = 30          -> 32 - 30 = 2
GROUND_TRUTH = "2"

COT_PROMPT = (
    "Solve the following word problem. Think step by step, showing your "
    "arithmetic explicitly. On the very last line, output ONLY the line:\n"
    "Final Answer: <number>\n\n"
    f"Problem: {QUESTION}"
)

FINAL_ANSWER_RE = re.compile(r"Final Answer:\s*([\-0-9.,]+)", re.IGNORECASE)


def extract_final_answer(text: str) -> str:
    """Pull the value following 'Final Answer:' out of a CoT response.

    Falls back to the last number-like token in the text if the model didn't
    follow the requested format exactly -- CoT outputs are free text, so a
    forgiving extractor makes the voting step more robust.
    """
    match = FINAL_ANSWER_RE.search(text)
    if match:
        return match.group(1).strip().rstrip(".").replace(",", "")
    numbers = re.findall(r"-?\d+(?:\.\d+)?", text)
    return numbers[-1] if numbers else "UNPARSEABLE"


In [ ]:
# ============ SINGLE-SAMPLE BASELINE ============
# This is what a naive "ask the LLM once" CoT agent does. At a non-zero
# temperature this single sample may land on a correct OR an incorrect
# reasoning path -- there's no mechanism here to catch a mistake.

baseline_llm = get_llm(temperature=0.7)
baseline_response = baseline_llm.invoke(COT_PROMPT)
baseline_text = baseline_response.content
baseline_answer = extract_final_answer(baseline_text)

print("--- Single-Sample Baseline ---")
print(baseline_text)
print(f"\nExtracted baseline answer: {baseline_answer}")
print(f"Ground truth: {GROUND_TRUTH}")
print(f"Baseline correct? {baseline_answer == GROUND_TRUTH}")


In [ ]:
# ============ SELF-CONSISTENCY: SAMPLE N INDEPENDENT REASONING PATHS ============
N_SAMPLES = 5

# A fresh LLM handle per call is not required (the client is stateless across
# .invoke calls), but we keep one shared, non-zero-temperature LLM and simply
# call it N times -- each call is an independent draw from the model's output
# distribution because temperature > 0 injects sampling randomness.
sampling_llm = get_llm(temperature=0.7)

samples = []
for i in range(N_SAMPLES):
    response = sampling_llm.invoke(COT_PROMPT)
    text = response.content
    answer = extract_final_answer(text)
    samples.append({"index": i + 1, "text": text, "answer": answer})

for s in samples:
    print(f"--- Sample {s['index']} (answer extracted: {s['answer']}) ---")
    print(s["text"])
    print()


In [ ]:
# ============ VOTE TALLY & CONSOLIDATION ============
vote_counts = Counter(s["answer"] for s in samples)
majority_answer, majority_votes = vote_counts.most_common(1)[0]

print("Vote tally across N =", N_SAMPLES, "samples:")
for answer, count in vote_counts.most_common():
    marker = "  <-- majority" if answer == majority_answer else ""
    print(f"  {answer!r}: {count} vote(s){marker}")

print(f"\nSelf-Consistency answer (majority vote): {majority_answer}")
print(f"Ground truth: {GROUND_TRUTH}")
print(f"Self-Consistency correct? {majority_answer == GROUND_TRUTH}")

print("\n--- Comparison ---")
print(f"Single-sample baseline : {baseline_answer} "
      f"({'correct' if baseline_answer == GROUND_TRUTH else 'INCORRECT'})")
print(f"Self-Consistency vote  : {majority_answer} "
      f"({'correct' if majority_answer == GROUND_TRUTH else 'INCORRECT'})")


### Discussion of the Output

Look at the `answer` extracted from each of the five samples above. Because we sampled at `temperature=0.7`, the five reasoning traces are not identical — some may add the baked trays before subtracting the morning sales, some may compute the afternoon sales before the morning sales, and one or two may make an arithmetic slip somewhere (e.g. forgetting the "twice as many" multiplier, or mis-adding `23 + 24`). Any single one of these traces, taken alone, is exactly the single-sample baseline above and could easily land on the wrong number.

What the vote tally shows is the core mechanism of Self-Consistency: **correct reasoning paths tend to converge on the same answer (`2` in this problem), while incorrect reasoning paths tend to disagree with each other** — one sample might say `17`, another `32`, another `47`, each from a different mistake. Because the correct answer is reproduced by independent, unrelated reasoning paths far more often than any single wrong answer is, the majority vote reliably surfaces `2` even when the single-sample baseline above happens to be wrong.

This is precisely why Self-Consistency is worth its extra `N`x cost on problems like this one: it converts the *possibility* of a correct single sample into a much higher *probability* of a correct consolidated answer, at the price of running the same prompt multiple times instead of engineering a smarter single prompt.

## Conclusion

In this notebook we implemented **Self-Consistency**: sample the same Chain-of-Thought prompt `N` times independently at a non-zero temperature, extract each sample's final answer, and consolidate via majority vote. On our bakery word problem, this converted a single, possibly-wrong CoT sample into a robust, verifiable consolidated answer, backed by agreement among independently reasoned paths.

Key takeaways:
- **No architecture change, no extra model** — Self-Consistency is pure decoding-time sampling plus a vote (`Counter` over extracted final answers).
- **Independence is the whole point.** Unlike a debate, reflection loop, or Ensemble-of-personas pattern, the `N` samples never see each other's reasoning — that's what makes their agreement statistically meaningful.
- **It only pays off when answers are extractable and comparable.** Numeric answers, multiple-choice letters, and short factual strings vote well; long free-form essays do not.
- **It corrects random errors, not systematic bias.** If the model consistently misreads the problem the same way, all `N` samples will agree on the same wrong answer — voting cannot fix that.

This notebook sits in **Layer B4 ("Sampling and search")** of this repo's agent-pattern grouping (`05_AI_Agent_Fundamentals/Agent_Pattern_Grouping.md`), alongside:
- **Tree of Thoughts** (`04_Tree_of_Thoughts.ipynb`) — branches and evaluates *intermediate* thoughts rather than only voting on final answers.
- **LATS** — combines tree search with reward-guided expansion (Monte Carlo Tree Search over reasoning/action steps).
- **Ensemble** (`08_Ensemble.ipynb`) — runs *differently-prompted* agents (personas) in parallel and synthesizes their outputs with an aggregator, rather than voting identical prompts.
- **Mental Loop** (`05_Mental_Loop.ipynb`) — simulates outcomes internally before acting, rather than sampling multiple final answers.

Self-Consistency is the simplest member of this family: no tree, no simulator, no aggregator model — just sample-and-vote. It is often the first thing worth trying before reaching for the heavier search-based patterns in this same layer.